# Salinas Spatial Benchmark Protocol

**Model-agnostic reproducibility protocol**

This notebook section is shared conceptually by SSFTT, Hybrid, and future comparison models. It is **not SSFTT-specific**.

The fixed benchmark protocol is:

- Salinas dataset
- spatial split only
- 32×32 spatial blocks
- class-aware block assignment
- 60/20/20 target split
- seed 42
- 15×15 centered patches
- original labeled-pixel indices
- no buffered/eroded indices
- training batch size 128
- evaluation batch size 128 for cross-model comparability
- AdamW: lr=1e-3, weight_decay=1e-4
- CrossEntropyLoss
- 50 epochs
- CosineAnnealingLR with T_max=50
- best checkpoint selected by validation Macro-F1

Generated benchmark artifacts use **generic names** so they can be reused by every model.

In [1]:
from pathlib import Path
import sys, os, json, hashlib, platform, subprocess, random, gc

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from dfm.data.salinas import (
    SALINAS_CLASS_NAMES,
    SalinasPatchDataset,
    load_salinas,
)

SEED = 42
PATCH_SIZE = 15
BLOCK_SIZE = 32
TRAIN_FRACTION = 0.60
VAL_FRACTION = 0.20
BATCH_SIZE = 128
EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

OUTPUTS = PROJECT_ROOT / "outputs" / "salinas"
OUTPUTS.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup
Device: cuda
GPU: NVIDIA GeForce RTX 2050


In [2]:
# Reproducibility controls.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed:", SEED)
print("CuDNN deterministic:", torch.backends.cudnn.deterministic)

Seed: 42
CuDNN deterministic: True


In [3]:
# Repository/environment provenance.
def git(cmd):
    try:
        return subprocess.check_output(
            ["git"] + cmd,
            cwd=PROJECT_ROOT,
            stderr=subprocess.STDOUT,
            text=True,
        ).strip()
    except Exception as e:
        return f"NOT FOUND IN SOURCES: {e}"

print("Git commit:", git(["rev-parse", "HEAD"]))
print("Git branch:", git(["branch", "--show-current"]))
print("Git status:", git(["status", "--short"]))
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

Git commit: a4178bee787808f159f712eb0bdd3d9155ffd29a
Git branch: main
Git status: M notebooks/04_hybrid_model.ipynb
 M notebooks/08_salinas_transformer_training.ipynb
 M notebooks/09_salinas_spatial_split_training.ipynb
 M notebooks/10_salinas_hybrid_ablation.ipynb
 M notebooks/12_salinas_ssftt_spatial.ipynb
 M notebooks/13_salinas_morphformer_spatial.ipynb
 M notebooks/15_salinas_gaht_spatial.ipynb
 D outputs/salinas/ssftt_official_spatial_best.pt
 D outputs/salinas/ssftt_official_spatial_history.csv
?? data/Salinas_corrected.mat
?? data/Salinas_gt.mat
?? data/salinas/Salinas_gt.mat
?? notebooks/01_salinas_spatial_benchmark_ssftt.ipynb
?? notebooks/02_salinas_spatial_benchmark_pca_hybrid.ipynb
?? notebooks/19_salinas_pca_ssftt_hybrid_spatial_training_FIXED_v2.ipynb
?? notebooks/data/
?? outputs/salinas/hybrid_pca30_spatial_spatial_best.pt
?? outputs/salinas/hybrid_pca30_spatial_spatial_history.csv
?? outputs/salinas/hybrid_pca30_spatial_spatial_run_config.json
?? outputs/salinas/hybri

In [4]:
data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"
scene = load_salinas(data_dir, download=True)

rows, cols = scene.labels.shape
labels = scene.labels

print("Cube:", scene.cube.shape)
print("Labels:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(SALINAS_CLASS_NAMES))

Cube: (512, 217, 204)
Labels: (512, 217)
Bands: 204
Classes: 16


In [5]:
def find_spatial_split(
    rows,
    cols,
    labels,
    block_size=32,
    train_fraction=0.60,
    val_fraction=0.20,
    seed=42,
    n_trials=10000,
):
    rng = np.random.default_rng(seed)

    block_rows = rows // block_size
    block_cols = cols // block_size

    block_keys = np.stack(
        [block_rows, block_cols],
        axis=1
    )

    unique_blocks = np.unique(block_keys, axis=0)

    n_blocks = len(unique_blocks)

    n_train = round(train_fraction * n_blocks)
    n_val = round(val_fraction * n_blocks)

    best_score = np.inf
    best_assignment = None

    # Precompute class counts for each block
    block_counts = []

    for br, bc in unique_blocks:

        mask = (
            (block_rows == br) &
            (block_cols == bc)
        )

        counts = np.bincount(
            labels[mask],
            minlength=len(SALINAS_CLASS_NAMES)
        )

        block_counts.append(counts)

    block_counts = np.asarray(block_counts)

    total_class_counts = block_counts.sum(axis=0)

    for _ in range(n_trials):

        permutation = rng.permutation(n_blocks)

        train_ids = permutation[:n_train]
        val_ids = permutation[n_train:n_train + n_val]
        test_ids = permutation[n_train + n_val:]

        train_counts = block_counts[train_ids].sum(axis=0)
        val_counts = block_counts[val_ids].sum(axis=0)
        test_counts = block_counts[test_ids].sum(axis=0)

        # Every class must occur in every split
        if np.any(train_counts == 0):
            continue

        if np.any(val_counts == 0):
            continue

        if np.any(test_counts == 0):
            continue

        # Compare class proportions with overall distribution
        overall_dist = total_class_counts / total_class_counts.sum()

        train_dist = train_counts / train_counts.sum()
        val_dist = val_counts / val_counts.sum()
        test_dist = test_counts / test_counts.sum()

        distribution_error = (
            np.mean(np.abs(train_dist - overall_dist))
            + np.mean(np.abs(val_dist - overall_dist))
            + np.mean(np.abs(test_dist - overall_dist))
        )

        # Compare number of pixels with desired 60/20/20
        total_pixels = len(labels)

        target_train = train_fraction * total_pixels
        target_val = val_fraction * total_pixels
        target_test = (1 - train_fraction - val_fraction) * total_pixels

        size_error = (
            abs(train_counts.sum() - target_train) / total_pixels
            + abs(val_counts.sum() - target_val) / total_pixels
            + abs(test_counts.sum() - target_test) / total_pixels
        )

        score = distribution_error + size_error

        if score < best_score:
            best_score = score

            best_assignment = (
                unique_blocks[train_ids].copy(),
                unique_blocks[val_ids].copy(),
                unique_blocks[test_ids].copy(),
            )

    if best_assignment is None:
        raise RuntimeError(
            "Could not find a valid spatial split. "
            "Try increasing n_trials or changing block_size."
        )

    train_blocks, val_blocks, test_blocks = best_assignment

    def blocks_to_indices(blocks):

        mask = np.zeros(len(rows), dtype=bool)

        for br, bc in blocks:
            mask |= (
                (block_rows == br) &
                (block_cols == bc)
            )

        return np.where(mask)[0]

    train_indices = blocks_to_indices(train_blocks)
    val_indices = blocks_to_indices(val_blocks)
    test_indices = blocks_to_indices(test_blocks)

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    rng.shuffle(test_indices)

    return (
        train_indices,
        val_indices,
        test_indices,
        train_blocks,
        val_blocks,
        test_blocks,
        best_score,
    )

In [6]:
SPLIT_PATH = OUTPUTS / "salinas_spatial_split_seed42.npz"

if SPLIT_PATH.exists():
    split = np.load(SPLIT_PATH)
    train_indices = split["train_indices"]
    val_indices = split["val_indices"]
    test_indices = split["test_indices"]
    print("Loaded existing spatial split:", SPLIT_PATH)
else:
    (
        train_indices,
        val_indices,
        test_indices,
        train_blocks,
        val_blocks,
        test_blocks,
        split_score,
    ) = find_spatial_split(
        rows,
        cols,
        labels,
        block_size=BLOCK_SIZE,
        train_fraction=TRAIN_FRACTION,
        val_fraction=VAL_FRACTION,
        seed=SEED,
        n_trials=10000,
    )
    np.savez(
        SPLIT_PATH,
        train_indices=train_indices,
        val_indices=val_indices,
        test_indices=test_indices,
    )
    print("Created spatial split:", SPLIT_PATH)
    print("Split score:", split_score)

print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Test:", len(test_indices))

Loaded existing spatial split: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\salinas_spatial_split_seed42.npz
Train: 32337
Validation: 10952
Test: 10840


In [7]:
def sha256_array(a):
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()

split_hashes = {
    "train": sha256_array(train_indices),
    "val": sha256_array(val_indices),
    "test": sha256_array(test_indices),
}

print(json.dumps(split_hashes, indent=2))

with open(OUTPUTS / "salinas_spatial_split_fingerprints.json", "w") as f:
    json.dump(split_hashes, f, indent=2)

{
  "train": "ec3617b38356b1ad4e2b8445b345f87be519a53ca34626484cb93d7b48a171d6",
  "val": "f569fc2e69cbf6bc770e170b9735459d7332d024fe06c433cde990c0f5287d8d",
  "test": "6fb69fcd80676361c70bb751f6f73361dfe8d083b655de33fa7d69424358617e"
}


In [8]:
train_dataset = SalinasPatchDataset(
    scene=scene, indices=train_indices, patch_size=PATCH_SIZE, normalize=True
)
val_dataset = SalinasPatchDataset(
    scene=scene, indices=val_indices, patch_size=PATCH_SIZE, normalize=True
)
test_dataset = SalinasPatchDataset(
    scene=scene, indices=test_indices, patch_size=PATCH_SIZE, normalize=True
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 32337
Validation: 10952
Test: 10840


# Common 30-component PCA preprocessing

PCA is a **generic benchmark preprocessing artifact**, not an SSFTT-specific artifact.

Canonical artifact:
`outputs/salinas/salinas_pca_30.joblib`

It is fitted **only from training patches** and then reused unchanged by every model that uses the PCA-30 input representation.

If the artifact already exists, it is loaded rather than refit.

In [9]:
from sklearn.decomposition import PCA
import joblib

PCA_COMPONENTS = 30
PCA_PATCHES = 2000
PCA_SPECTRAL_VECTORS = 200_000

PCA_PATH = OUTPUTS / "salinas_pca_30.joblib"
PCA_CONFIG_PATH = OUTPUTS / "salinas_pca_30_config.json"

if PCA_PATH.exists() and PCA_CONFIG_PATH.exists():
    pca = joblib.load(PCA_PATH)
    with open(PCA_CONFIG_PATH) as f:
        pca_config = json.load(f)

    if getattr(pca, "n_components_", None) != PCA_COMPONENTS:
        raise ValueError("Existing PCA artifact is not PCA-30.")

    print("Loaded existing common PCA:", PCA_PATH)
else:
    rng = np.random.default_rng(SEED)

    n_selected = min(PCA_PATCHES, len(train_dataset))
    selected_indices = rng.choice(
        len(train_dataset),
        size=n_selected,
        replace=False,
    )

    sampled_patches = np.stack([
        train_dataset[int(i)][0].numpy()
        for i in selected_indices
    ])

    sampled_spectra = (
        sampled_patches.transpose(0, 2, 3, 1)
        .reshape(-1, sampled_patches.shape[1])
    )

    if len(sampled_spectra) > PCA_SPECTRAL_VECTORS:
        keep = rng.choice(
            len(sampled_spectra),
            size=PCA_SPECTRAL_VECTORS,
            replace=False,
        )
        sampled_spectra = sampled_spectra[keep]

    pca = PCA(
        n_components=PCA_COMPONENTS,
        svd_solver="randomized",
        random_state=SEED,
    )
    pca.fit(sampled_spectra)

    pca_config = {
        "artifact": "salinas_pca_30.joblib",
        "components": PCA_COMPONENTS,
        "patches_sampled": n_selected,
        "spectral_vectors_used": int(len(sampled_spectra)),
        "svd_solver": "randomized",
        "random_state": SEED,
        "fit_source": "training patches only",
    }

    joblib.dump(pca, PCA_PATH)
    with open(PCA_CONFIG_PATH, "w") as f:
        json.dump(pca_config, f, indent=2)

    print("Created common PCA:", PCA_PATH)

    del sampled_patches, sampled_spectra
    gc.collect()

print("Components:", pca.n_components_)
print("Explained variance:", float(pca.explained_variance_ratio_.sum()))

Loaded existing common PCA: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\salinas_pca_30.joblib
Components: 30
Explained variance: 0.9995962977409363


In [10]:
class PCAPatchDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, pca, add_channel_dim=False):
        self.base_dataset = base_dataset
        self.pca = pca
        self.add_channel_dim = add_channel_dim

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        patch, label = self.base_dataset[idx]
        x = patch  # [204,H,W]
        C, H, W = x.shape

        z = self.pca.transform(
            x.transpose(1, 2, 0).reshape(-1, C)
        )
        z = z.reshape(H, W, -1).transpose(2, 0, 1).astype(np.float32)

        if self.add_channel_dim:
            z = np.expand_dims(z, axis=0)

        return (
            torch.from_numpy(z),
            torch.tensor(label, dtype=torch.long),
        )

# PCA-30 Hybrid spatial experiment

**Only one architectural/input change is made to the existing Hybrid:**

`204 spectral input channels → common PCA-30 representation → Hybrid`

The Hybrid architecture remains otherwise unchanged:

- original SpatialCNNBranch
- original SpectralAttentionBranch
- original gated fusion
- feature_dim=128
- spectral depth=2
- spectral heads=4
- dropout=0.1
- max_bands=256

In [11]:
from dfm.models.hybrid import HybridSpatialSpectralClassifier

pca_train_dataset = PCAPatchDataset(train_dataset, pca, add_channel_dim=False)
pca_val_dataset = PCAPatchDataset(val_dataset, pca, add_channel_dim=False)
pca_test_dataset = PCAPatchDataset(test_dataset, pca, add_channel_dim=False)

train_loader_current = DataLoader(
    pca_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
val_loader_current = DataLoader(
    pca_val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
test_loader_current = DataLoader(
    pca_test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

MODEL_NAME = "hybrid_pca30_spatial"

model = HybridSpatialSpectralClassifier(
    in_channels=30,
    num_classes=len(SALINAS_CLASS_NAMES),
    feature_dim=128,
    spectral_depth=2,
    spectral_heads=4,
    fusion="gated",
    dropout=0.1,
    max_bands=256,
).to(device)

print(model)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

x0, y0 = next(iter(train_loader_current))
print("Hybrid input batch:", x0.shape)

HybridSpatialSpectralClassifier(
  (spatial): SpatialCNNBranch(
    (net): Sequential(
      (0): Conv2d(30, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): GELU(approximate='none')
      (9): AdaptiveAvgPool2d(output_size=1)
      (10): Flatten(start_dim=1, end_dim=-1)
    )
  )
  (spectral): SpectralAttentionBranch(
    (value_projection): Linear(in_features=1, out_features=128, bias=True)
    (band_embedding): Embedding(256, 128)
    (encoder): TransformerEncoder(
      (layers): ModuleLis

In [12]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total = 0
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in tqdm(loader, leave=False):
            x = x.to(device=device, dtype=torch.float32)
            y = y.to(device=device, dtype=torch.long)

            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * y.size(0)
            total += y.size(0)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(logits.argmax(dim=1).cpu().numpy())

    from sklearn.metrics import accuracy_score, f1_score

    return {
        "loss": total_loss / total,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }, np.asarray(y_true), np.asarray(y_pred)

In [13]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    total = 0

    for x, y in tqdm(loader, leave=False):
        x = x.to(device=device, dtype=torch.float32)
        y = y.to(device=device, dtype=torch.long)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        total += y.size(0)

    return running_loss / total

In [14]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
)

best_macro_f1 = -1.0
best_epoch = 0
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model, train_loader_current, criterion, optimizer
    )

    val_metrics, _, _ = evaluate(
        model, val_loader_current, criterion
    )

    scheduler.step()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "lr": optimizer.param_groups[0]["lr"],
    })

    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss {train_loss:.4f} | "
        f"Val Acc {val_metrics['accuracy']:.4f} | "
        f"Val Macro-F1 {val_metrics['macro_f1']:.4f}"
    )

print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", best_macro_f1)

Epoch 01/50 | Loss 0.2051 | Val Acc 0.8509 | Val Macro-F1 0.8531


Epoch 02/50 | Loss 0.0176 | Val Acc 0.9396 | Val Macro-F1 0.8795


Epoch 03/50 | Loss 0.0127 | Val Acc 0.8447 | Val Macro-F1 0.8507


Epoch 04/50 | Loss 0.0088 | Val Acc 0.8557 | Val Macro-F1 0.8558


Epoch 05/50 | Loss 0.0071 | Val Acc 0.8962 | Val Macro-F1 0.8942


...

Epoch 46/50 | Loss 0.0000 | Val Acc 0.9158 | Val Macro-F1 0.9074


Epoch 47/50 | Loss 0.0000 | Val Acc 0.9171 | Val Macro-F1 0.9017


Epoch 48/50 | Loss 0.0000 | Val Acc 0.9228 | Val Macro-F1 0.9119


Epoch 49/50 | Loss 0.0000 | Val Acc 0.9170 | Val Macro-F1 0.8996


Epoch 50/50 | Loss 0.0000 | Val Acc 0.9180 | Val Macro-F1 0.9008
Best epoch: 13
Best validation Macro-F1: 0.922670883164553


In [15]:
model.load_state_dict(best_state)
model.to(device)

test_metrics, y_test, y_pred = evaluate(
    model, test_loader_current, criterion
)

print("=" * 60)
print("FINAL SPATIAL TEST RESULTS")
print("=" * 60)
print(f"Loss     : {test_metrics['loss']:.6f}")
print(f"Accuracy : {test_metrics['accuracy']:.6f}")
print(f"Macro-F1 : {test_metrics['macro_f1']:.6f}")

  0%|          | 0/85 [00:00<?, ?it/s]

FINAL SPATIAL TEST RESULTS
Loss     : 0.117894
Accuracy : 0.970295
Macro-F1 : 0.948041


In [16]:
history_df = pd.DataFrame(history)
history_df.to_csv(
    OUTPUTS / f"{MODEL_NAME}_spatial_history.csv",
    index=False,
)

torch.save(
    {
        "model_state_dict": best_state,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_macro_f1,
        "test_metrics": test_metrics,
        "seed": SEED,
        "split_hashes": split_hashes,
        "patch_size": PATCH_SIZE,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
    },
    OUTPUTS / f"{MODEL_NAME}_spatial_best.pt",
)

np.save(
    OUTPUTS / f"{MODEL_NAME}_spatial_y_test.npy",
    y_test,
)
np.save(
    OUTPUTS / f"{MODEL_NAME}_spatial_y_pred.npy",
    y_pred,
)

run_config = {
    "model": MODEL_NAME,
    "seed": SEED,
    "patch_size": PATCH_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "CosineAnnealingLR(T_max=50)",
    "split_artifact": "salinas_spatial_split_seed42.npz",
    "split_hashes": split_hashes,
    "pca_artifact": "salinas_pca_30.joblib",
}

with open(
    OUTPUTS / f"{MODEL_NAME}_spatial_run_config.json", "w"
) as f:
    json.dump(run_config, f, indent=2)

print("Saved reproducibility artifacts.")

Saved reproducibility artifacts.


In [19]:
# ============================================================
# Efficiency Benchmark — PCA Hybrid Spatial Model
# ============================================================

import time
import os
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. Model parameters and size
# ------------------------------------------------------------
model.eval()

parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

parameters_m = parameters / 1e6
model_size_mb = (
    sum(p.numel() * p.element_size() for p in model.parameters())
    / (1024 ** 2)
)

print(f"Trainable parameters : {parameters:,}")
print(f"Parameters (M)       : {parameters_m:.6f}")
print(f"Model size (MB)      : {model_size_mb:.4f}")


# ------------------------------------------------------------
# 2. FLOPs
# ------------------------------------------------------------
flops = float("nan")
gflops = float("nan")

try:
    from fvcore.nn import FlopCountAnalysis

    # Get one representative batch
    x_example, _ = next(iter(test_loader_current))
    x_example = x_example.to(device=device, dtype=torch.float32)

    model_for_flops = model.to(device)
    model_for_flops.eval()

    with torch.no_grad():
        flop_analysis = FlopCountAnalysis(
            model_for_flops,
            x_example
        )

    flops = float(flop_analysis.total())
    gflops = flops / 1e9

    print(f"FLOPs / batch        : {flops:,.0f}")
    print(f"GFLOPs / batch       : {gflops:.6f}")

except Exception as e:
    print("FLOPs not available.")
    print("Install fvcore with: pip install fvcore")
    print("Reason:", e)


# ------------------------------------------------------------
# 3. GPU memory benchmark
# ------------------------------------------------------------
peak_gpu_memory_mb = float("nan")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

    # One complete evaluation pass
    with torch.no_grad():
        for x, y in test_loader_current:
            x = x.to(device=device, dtype=torch.float32)
            _ = model(x)

    torch.cuda.synchronize()

    peak_gpu_memory_mb = (
        torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    )

    print(f"Peak GPU memory     : {peak_gpu_memory_mb:.4f} MB")


# ------------------------------------------------------------
# 4. Latency / throughput
# ------------------------------------------------------------
LATENCY_WARMUP = 20
LATENCY_RUNS = 100

# Get one fixed batch
x_bench, _ = next(iter(test_loader_current))
x_bench = x_bench.to(device=device, dtype=torch.float32)

model.eval()

# Warm-up
with torch.no_grad():
    for _ in range(LATENCY_WARMUP):
        _ = model(x_bench)

if torch.cuda.is_available():
    torch.cuda.synchronize()

# Timed runs
batch_times = []

with torch.no_grad():
    for _ in range(LATENCY_RUNS):

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        _ = model(x_bench)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.perf_counter()

        batch_times.append((end - start) * 1000.0)


batch_times = torch.tensor(batch_times, dtype=torch.float64)

latency_batch_ms = batch_times.mean().item()
latency_batch_median_ms = batch_times.median().item()
latency_batch_p95_ms = torch.quantile(batch_times, 0.95).item()
latency_batch_std_ms = batch_times.std(unbiased=True).item()

batch_size_bench = x_bench.size(0)

latency_per_sample_ms = (
    latency_batch_ms / batch_size_bench
)

throughput_samples_sec = (
    batch_size_bench / (latency_batch_ms / 1000.0)
)

print(f"Batch size          : {batch_size_bench}")
print(f"Mean latency        : {latency_batch_ms:.4f} ms")
print(f"Median latency      : {latency_batch_median_ms:.4f} ms")
print(f"P95 latency         : {latency_batch_p95_ms:.4f} ms")
print(f"Latency/sample      : {latency_per_sample_ms:.6f} ms")
print(f"Throughput          : {throughput_samples_sec:.2f} samples/s")


# ------------------------------------------------------------
# 5. Test accuracy / Macro-F1
# ------------------------------------------------------------
test_metrics, y_true, y_pred = evaluate(
    model,
    test_loader_current,
    criterion
)

accuracy = test_metrics["accuracy"]
macro_f1 = test_metrics["macro_f1"]

print(f"Test Accuracy       : {accuracy:.6f}")
print(f"Test Macro-F1       : {macro_f1:.6f}")


# ------------------------------------------------------------
# 6. Create benchmark record
# ------------------------------------------------------------
benchmark_row = {
    "model": MODEL_NAME,
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "parameters": parameters,
    "parameters_m": parameters_m,
    "model_size_mb": model_size_mb,
    "flops": flops,
    "gflops": gflops,
    "peak_gpu_memory_mb": peak_gpu_memory_mb,
    "batch_size": batch_size_bench,
    "latency_batch_ms": latency_batch_ms,
    "latency_batch_median_ms": latency_batch_median_ms,
    "latency_batch_p95_ms": latency_batch_p95_ms,
    "latency_batch_std_ms": latency_batch_std_ms,
    "latency_per_sample_ms": latency_per_sample_ms,
    "throughput_samples_sec": throughput_samples_sec,
}

benchmark_df = pd.DataFrame([benchmark_row])

display(benchmark_df)


# ------------------------------------------------------------
# 7. Save benchmark
# ------------------------------------------------------------
OUTPUT_DIR = "outputs/salinas"
os.makedirs(OUTPUT_DIR, exist_ok=True)

benchmark_path = os.path.join(
    OUTPUT_DIR,
    f"{MODEL_NAME}_efficiency_final.csv"
)

benchmark_df.to_csv(
    benchmark_path,
    index=False
)

print("\nSaved:", benchmark_path)

Trainable parameters : 594,576
Parameters (M)       : 0.594576
Model size (MB)      : 2.2681
FLOPs not available.
Install fvcore with: pip install fvcore
Reason: No module named 'fvcore'
Peak GPU memory     : 60.3408 MB
Batch size          : 128
Mean latency        : 8.1535 ms
Median latency      : 8.0986 ms
P95 latency         : 8.5083 ms
Latency/sample      : 0.063699 ms
Throughput          : 15698.75 samples/s


Test Accuracy       : 0.970295
Test Macro-F1       : 0.948041


,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,hybrid_pca30_spatial,0.970295,0.948041,594576,0.594576,2.268127,NaN,NaN,60.34082,128,8.153514,8.0986,8.50829,0.204313,0.063699,15698.752699



Saved: outputs/salinas\hybrid_pca30_spatial_efficiency_final.csv
